## Executive Summary

This notebook computes **winding factors** — the efficiency with which a winding produces each spatial harmonic of the airgap field.

| Function | What it computes |
|----------|------------------|
| `pitch_factor` | kp(ν) — chording factor |
| `distribution_factor` | kd(ν) — belt factor (integer-slot only) |
| `winding_factor` | kw(ν) = kp·kd or phasor sum |
| `winding_factor_spectrum` | |kw(ν)| for ν = 1…ν_max |

## How It Fits Into Motor Design

```
01_winding_sos.ipynb  (layout engine)
      │
      ▼
01_winding_factors.ipynb   ←── YOU ARE HERE
      │
      ▼
01_winding_mmf.ipynb       (MMF waveform & spectrum)
```

**Depends on:** `sos.py` — uses `build_coil_matrix`, `winding_factor_sos`, and `_resolve_layout` to fill in a missing `layers` or `w`.

**Feeds into:** `01_winding_mmf.ipynb` (via `winding_factor_spectrum`).

## Notation

| Symbol | Definition |
|--------|------------|
| Q | Stator slots |
| P | Poles |
| p | Pole pairs = P/2 |
| τp | Pole pitch = Q/P (one magnetic pole) |
| w | Coil span in slots |
| ν | Mechanical harmonic order |
| kp | Pitch (chording) factor |
| kd | Distribution (belt) factor |
| kw | Combined winding factor = kp·kd (ISW) |
| q | Slots per pole per phase = Q/(2mp) |

## Imports and Setup

In [ ]:
#| default_exp winding.winding_factors

In [ ]:
#| export
from __future__ import annotations

from fractions import Fraction

import numpy as np

from emachines.winding.sos import _resolve_layout, build_coil_matrix, winding_factor_sos

__all__ = [
    "pitch_factor",
    "distribution_factor",
    "winding_factor",
    "winding_factor_spectrum",
]

---

## `pitch_factor`: Chording Factor kp

### Theory

All harmonic orders ν in this module are **mechanical**: ν cycles per revolution, so the working harmonic is ν = p. A coil spanning w of the Q slots links the ν-th harmonic with

$$k_{p\nu} = \left|\sin\!\left(\frac{\nu \pi w}{Q}\right)\right|$$

This is the textbook $\sin(\nu' \tfrac{\pi}{2} \tfrac{w}{\tau_p})$ with electrical order ν' = ν/p and pole pitch τp = Q/P. It is a property of one coil, so it holds for any ν. For 12s/4p the full pitch is w = 3 slots, and the working harmonic is ν = 2.

**References:** Hanselman (2003), eq. 4.5; Pyrhönen et al. (2014), §2.4.

In [ ]:
#| export
def pitch_factor(nu: int, *, Q: int, w: int) -> float:
    """
    Pitch (chording) factor kp of one coil for mechanical harmonic nu.

    kp(nu) = |sin(nu * pi * w / Q)|

    Parameters
    ----------
    nu : int   Mechanical harmonic order (working harmonic nu = p)
    Q  : int   Number of stator slots (keyword-only)
    w  : int   Coil span in slots (keyword-only)

    Returns
    -------
    float   kp in [0, 1]

    Examples
    --------
    >>> pitch_factor(2, Q=24, w=6)   # 24s/4p, full pitch, working harmonic
    1.0
    >>> round(pitch_factor(2, Q=24, w=5), 4)   # 5/6 chording
    0.9659
    """
    return float(np.abs(np.sin(nu * np.pi * w / Q)))


In [ ]:
from emachines.winding.winding_factors import pitch_factor

# 24s/4p: p = 2, pole pitch = Q/P = 6 slots. Working harmonic ν = 2,
# 5th and 7th electrical harmonics are ν = 10 and 14.
Q, p = 24, 2
print("Pitch factor for 24s/4p (pole pitch = 6 slots):")
print(f"{'w':>4}  {'kp(ν=2)':>8}  {'kp(ν=10)':>9}  {'kp(ν=14)':>9}")
print("-" * 38)
for w in range(1, 7):
    marker = " <- full pitch" if w == 6 else (" <- 5/6 chording" if w == 5 else "")
    print(f"  {w:>2}  {pitch_factor(2, Q=Q, w=w):>8.4f}  {pitch_factor(10, Q=Q, w=w):>9.4f}  "
          f"{pitch_factor(14, Q=Q, w=w):>9.4f}{marker}")

---

## `distribution_factor`: Belt Factor kd

### Theory

Spreading a phase belt over q slots (q = Q/(2mp), slots per pole per phase) gives, in mechanical order ν,

$$k_{d\nu} = \left|\frac{\sin(\nu q \pi / Q)}{q \sin(\nu \pi / Q)}\right|$$

which is the textbook $\sin(\nu' \pi / 2m) \,/\, (q \sin(\nu' \pi / 2mq))$ with ν' = ν/p.

**Where the closed form is valid.** kd describes one phase belt. The full winding also repeats every pole pair, which cancels every ν that is not a multiple of p, and its go and return belts are mirror images, which cancels even electrical orders. So kp·kd equals the true winding factor only for **whole-number q** and **ν an odd multiple of p**. Elsewhere the function raises `ValueError` instead of returning a factor for a harmonic that does not exist. Use `winding_factor` for every other case.

**References:** Pyrhönen et al. (2014), eq. 2.15.

In [ ]:
#| export
def distribution_factor(nu: int, *, Q: int, P: int, m: int = 3) -> float:
    """
    Distribution (belt) factor kd for mechanical harmonic nu.

    kd(nu) = |sin(nu*q*pi/Q) / (q * sin(nu*pi/Q))|,   q = Q / (2*m*p)

    Valid only for whole-number q and nu an odd multiple of p, where
    kp*kd equals the winding factor. Use ``winding_factor`` otherwise.

    Parameters
    ----------
    nu : int   Mechanical harmonic order (working harmonic nu = p)
    Q  : int   Number of stator slots (keyword-only)
    P  : int   Number of poles (keyword-only)
    m  : int   Number of phases (default 3)

    Returns
    -------
    float   kd in (0, 1]

    Raises
    ------
    ValueError
        If q is not a whole number, or nu is not an odd multiple of p.

    Examples
    --------
    >>> round(distribution_factor(2, Q=12, P=4), 4)   # 12s/4p, q=1
    1.0
    >>> round(distribution_factor(2, Q=24, P=4), 4)   # 24s/4p, q=2
    0.9659
    """
    p = P // 2
    q = Fraction(Q, 2 * m * p)
    if q.denominator != 1:
        raise ValueError(
            f"distribution_factor() needs a whole-number q (slots per pole per phase); "
            f"got q = {q} for Q={Q}, P={P}, m={m}. Use winding_factor() instead.")
    if nu % p or (nu // p) % 2 == 0:
        raise ValueError(
            f"distribution_factor() is valid only for nu an odd multiple of p={p}; "
            f"got nu={nu}. That harmonic cancels in the full winding. "
            f"Use winding_factor() instead.")
    qi = int(q)
    den = qi * np.sin(nu * np.pi / Q)
    if np.abs(den) < 1e-12:
        return 1.0
    return float(np.abs(np.sin(nu * qi * np.pi / Q) / den))


In [ ]:
from emachines.winding.winding_factors import distribution_factor

print("Distribution factor at the working harmonic ν = p, 4 poles:")
print(f"{'Config':>16}  {'q':>3}  {'kd(ν=2)':>8}  {'kd(ν=10)':>9}")
print("-" * 42)
for Q in (12, 24, 36, 48):
    q = Q // 12
    print(f"  {Q:>2}s/4p        {q:>3}  {distribution_factor(2, Q=Q, P=4):>8.4f}  "
          f"{distribution_factor(10, Q=Q, P=4):>9.4f}")

for Q, P, nu in [(36, 8, 4), (24, 4, 4)]:
    try:
        distribution_factor(nu, Q=Q, P=P)
    except ValueError as e:
        print(f"\n{Q}s/{P}p, ν={nu} → ValueError: {e}")

---

## `winding_factor`: Combined kw

### Theory

`winding_factor(nu, Q=…, P=…)` is the exact winding factor of harmonic ν (mechanical order, working harmonic ν = p), from the star-of-slots phasor sum. It works for integer-slot, fractional-slot and tooth-coil windings, and for every ν, including sub-harmonics and orders that cancel (which give 0).

It describes the same winding as the rest of the module: when `layers` or `w` is left out, the best candidate from `design_windings` is used.

For whole-number q and ν an odd multiple of p, it equals the textbook product `pitch_factor(ν, Q=Q, w=w) * distribution_factor(ν, Q=Q, P=P)` for a double-layer winding of span w. The tests check this.

In [ ]:
#| export
def winding_factor(
    nu: int,
    *,
    Q: int,
    P: int,
    w: int | None = None,
    layers: int | None = None,
    m: int = 3,
) -> float:
    """
    Winding factor |kw| of mechanical harmonic nu, by the star of slots.

    Parameters
    ----------
    nu     : int          Mechanical harmonic order (working harmonic nu = p)
    Q      : int          Number of stator slots (keyword-only)
    P      : int          Number of poles (keyword-only)
    w      : int | None   Double-layer coil span in slots. None -> the best span.
    layers : int | None   1 or 2. None -> the best buildable, balanced candidate
                          (see ``design_windings``); a given w implies 2.
    m      : int          Number of phases (default 3)

    Returns
    -------
    float   |kw| in [0, 1]

    Examples
    --------
    >>> round(winding_factor(5, Q=12, P=10, w=1), 4)   # 12s/10p tooth coil, working harmonic
    0.933
    >>> round(winding_factor(2, Q=24, P=4, w=5), 4)    # 24s/4p, 5/6 chording
    0.933
    >>> round(winding_factor(4, Q=36, P=8, w=4), 4)    # 36s/8p, q = 3/2
    0.9452
    """
    if w is not None and layers is None:
        layers = 2
    return winding_factor_sos(nu, Q, P, m, layers=layers, w=w)


In [ ]:
from emachines.winding.winding_factors import winding_factor

configs = [
    dict(Q=12, P=4,  w=3, label="12s/4p  q=1    full pitch"),
    dict(Q=24, P=4,  w=5, label="24s/4p  q=2    5/6 chording"),
    dict(Q=36, P=8,  w=4, label="36s/8p  q=3/2  fractional"),
    dict(Q=12, P=10, w=1, label="12s/10p q=2/5  tooth coil"),
    dict(Q=12, P=8,  w=1, label="12s/8p  q=1/2  tooth coil"),
]
print(f"{'Configuration':<30}  {'p':>2}  {'kw(ν=p)':>8}  {'kw(ν=1)':>8}  {'kw(ν=5p)':>8}")
print("-" * 66)
for c in configs:
    p = c['P'] // 2
    kw = [winding_factor(nu, Q=c['Q'], P=c['P'], w=c['w']) for nu in (p, 1, 5 * p)]
    print(f"  {c['label']:<28}  {p:>2}  {kw[0]:>8.4f}  {kw[1]:>8.4f}  {kw[2]:>8.4f}")

---

## `winding_factor_spectrum`: Full Harmonic Spectrum

### Theory

The phasor sum method computes |kw(ν)| for all mechanical harmonics:

$$k_{w\nu} = \frac{1}{m} \sum_{k=0}^{m-1} \frac{\left|\sum_{i,\text{lyr}} s_{i,\text{lyr}} e^{j\nu 2\pi i/Q}\right|}{N_k}$$

Works for all winding types. **Working harmonic:** ν = p = P/2.

**References:** Müller, Vogt & Ponick (2008), eq. 3.65.

In [ ]:
#| export
def winding_factor_spectrum(
    Q: int,
    P: int,
    m: int = 3,
    layers: int | None = None,
    w: int | None = None,
    nu_max: int = 40,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Winding factor magnitude spectrum for mechanical harmonics 1...nu_max.

    Parameters
    ----------
    Q      : int   Number of stator slots
    P      : int   Number of poles (even, >= 2)
    m      : int   Number of phases (default 3)
    layers : int | None
             1 (single-layer) or 2 (double-layer).
             None -> the best buildable, balanced candidate (see design_windings).
    w      : int | None
             Double-layer coil span in slots. None -> the best span.
    nu_max : int   Highest mechanical harmonic (default 40)

    Returns
    -------
    nu_arr : np.ndarray, shape (nu_max,), dtype int
    kw_arr : np.ndarray, shape (nu_max,), dtype float64
        |kw(nu)| in [0, 1]

    Examples
    --------
    >>> nu, kw = winding_factor_spectrum(12, 10, layers=2)
    >>> round(float(kw[4]), 4)   # nu=5 working harmonic for 12s/10p
    0.933
    >>> nu, kw = winding_factor_spectrum(12, 4, layers=1, w=6, nu_max=10)
    >>> round(float(kw[1]), 4)   # nu=2, 12s/4p q=1 full-pitch
    1.0
    """
    layers, w = _resolve_layout(Q, P, m, layers, w)

    matrix = build_coil_matrix(Q, P, m, layers, w)
    slots = np.arange(Q, dtype=np.float64)
    nu_arr = np.arange(1, nu_max + 1, dtype=int)
    kw_arr = np.zeros(len(nu_arr), dtype=np.float64)

    for idx, nu in enumerate(nu_arr):
        phasors = np.exp(1j * nu * 2.0 * np.pi * slots / Q)
        kw_vals = []
        for k in range(m):
            ph = k + 1
            total = 0j
            n_cond = 0
            for lyr in range(layers):
                fwd = matrix[lyr] == +ph
                ret = matrix[lyr] == -ph
                total += phasors[fwd].sum() - phasors[ret].sum()
                n_cond += int(fwd.sum()) + int(ret.sum())
            if n_cond > 0:
                kw_vals.append(abs(total) / n_cond)
        kw_arr[idx] = float(np.mean(kw_vals)) if kw_vals else 0.0

    return nu_arr, kw_arr

In [ ]:
from emachines.winding.winding_factors import winding_factor_spectrum
import numpy as np

configs = [
    dict(Q=12, P=10, layers=2, label="12s/10p FSCW DL"),
    dict(Q=12, P=8,  layers=2, label="12s/8p  FSCW DL"),
    dict(Q=24, P=4,  layers=2, label="24s/4p  ISW  DL"),
]

for cfg in configs:
    nu, kw = winding_factor_spectrum(cfg['Q'], cfg['P'], layers=cfg['layers'], nu_max=15)
    p = cfg['P'] // 2
    print(f"\n{cfg['label']}  (working nu = {p})")
    for i in range(len(nu)):
        if kw[i] > 0.01:
            marker = " <- working" if nu[i] == p else ""
            print(f"  nu={int(nu[i]):>3}  kw={float(kw[i]):>6.4f}{marker}")

---

## References

1. Hanselman, D.C. (2003). *Brushless Permanent Magnet Motor Design*, 2nd ed. eq. 4.5.
2. Pyrhönen, J., Jokinen, T., & Hrabovcová, V. (2008). *Design of Rotating Electrical Machines*. Wiley. eq. 2.15.
3. Müller, G., Vogt, K., & Ponick, B. (2008). *Berechnung elektrischer Maschinen*. Wiley-VCH. eq. 3.65.
4. Bianchi, N., & Bolognani, S. (2002). IEEE Trans. Ind. Appl., 38(5). DOI: 10.1109/TIA.2002.802909.

---

## Tests

In [ ]:
#| hide
import math
import numpy as np
from emachines.winding.winding_factors import (
    pitch_factor, distribution_factor, winding_factor,
    winding_factor_spectrum,
)

# pitch_factor (mechanical order)
assert math.isclose(pitch_factor(2, Q=24, w=6), 1.0), "full pitch kp=1 at working harmonic"
assert math.isclose(pitch_factor(2, Q=24, w=5), 0.9659, abs_tol=1e-4), "5/6 chording"
assert math.isclose(pitch_factor(6, Q=24, w=4), 0.0, abs_tol=1e-10), "2/3 pitch cancels 3rd electrical"
print("pass pitch_factor")

# distribution_factor (mechanical order, valid range only)
assert math.isclose(distribution_factor(2, Q=12, P=4), 1.0, abs_tol=1e-10), "q=1 kd=1"
assert math.isclose(distribution_factor(2, Q=24, P=4), 0.9659, abs_tol=1e-4), "q=2"
for kwargs, nu, msg in [(dict(Q=12, P=10), 5, "whole-number q"),
                        (dict(Q=36, P=8), 4, "whole-number q"),
                        (dict(Q=24, P=4), 4, "odd multiple"),
                        (dict(Q=24, P=4), 3, "odd multiple")]:
    try:
        distribution_factor(nu, **kwargs)
        raise AssertionError(f"{kwargs} nu={nu} should raise")
    except ValueError as e:
        assert msg in str(e), str(e)
print("pass distribution_factor")

# winding_factor (mechanical order, star of slots)
assert math.isclose(winding_factor(2, Q=12, P=4, w=3), 1.0, abs_tol=1e-10)
assert math.isclose(winding_factor(5, Q=12, P=10, w=1), 0.9330, abs_tol=1e-3)  # nu=p=5 working harmonic
assert math.isclose(winding_factor(4, Q=12, P=8, w=1), 0.8660, abs_tol=1e-3)   # nu=p=4 working harmonic
assert math.isclose(winding_factor(1, Q=12, P=10, w=1), 0.0670, abs_tol=1e-3)  # sub-harmonic, not kw1
assert math.isclose(winding_factor(2, Q=24, P=4, w=5), 0.9330, abs_tol=1e-4)   # was nu=1 before
assert winding_factor(4, Q=24, P=4, w=5) < 1e-12, "2nd electrical cancels (closed form gave 0.433)"
assert math.isclose(winding_factor(4, Q=36, P=8, w=4), 0.9452, abs_tol=1e-4)   # q=3/2 (closed form gave 0.9598)
assert math.isclose(winding_factor(2, Q=15, P=4, w=3), 0.9099, abs_tol=1e-4)   # q=5/4 (closed form gave 0.9353)
try:
    winding_factor(5, 12, 5, 1)  # old positional call must fail, not reinterpret p as P
    raise AssertionError("positional Q/P should raise TypeError")
except TypeError:
    pass
print("pass winding_factor")

# Closed form = exact winding factor where it is valid
for Q, P in [(12, 4), (24, 4), (36, 4), (48, 4), (18, 2), (36, 6), (48, 8)]:
    p = P // 2
    for w in range(1, Q // P + 1):
        for nu in range(p, 41, 2 * p):           # odd multiples of p
            cf = pitch_factor(nu, Q=Q, w=w) * distribution_factor(nu, Q=Q, P=P)
            ex = winding_factor(nu, Q=Q, P=P, w=w)
            assert math.isclose(cf, ex, abs_tol=1e-12), (Q, P, w, nu, cf, ex)
print("pass kp*kd equals winding_factor for whole-number q at odd multiples of p")

# winding_factor agrees with winding_factor_spectrum for every nu and winding
for Q, P, layers, w in [(24, 4, 2, 5), (36, 8, 2, 4), (15, 4, 2, 3), (12, 10, 1, None),
                        (12, 10, 2, 1), (9, 8, 2, 1), (24, 4, None, None), (15, 8, None, None)]:
    _, spec = winding_factor_spectrum(Q, P, layers=layers, w=w, nu_max=40)
    for nu in range(1, 41):
        assert math.isclose(winding_factor(nu, Q=Q, P=P, layers=layers, w=w), spec[nu - 1], abs_tol=1e-12)
print("pass winding_factor matches winding_factor_spectrum")

# Defaults: layers / w left out → best candidate (see the design notebook)
_, kw_def = winding_factor_spectrum(15, 8, nu_max=4)
_, kw_w2 = winding_factor_spectrum(15, 8, layers=2, w=2, nu_max=4)
assert np.allclose(kw_def, kw_w2), "15s/8p default should be double layer, w=2 (kw1=0.951)"
print("pass winding_factor_spectrum defaults")

# winding_factor_spectrum
nu, kw = winding_factor_spectrum(12, 10, layers=2, nu_max=20)
assert nu.shape == (20,) and kw.shape == (20,)
assert math.isclose(float(kw[4]), 0.9330, abs_tol=1e-3)
assert np.all(kw >= -1e-10) and np.all(kw <= 1.0 + 1e-10)

_, kw8 = winding_factor_spectrum(12, 8, layers=2, nu_max=10)
assert math.isclose(float(kw8[3]), 0.8660, abs_tol=1e-3)

_, kw12 = winding_factor_spectrum(12, 4, layers=1, w=6, nu_max=10)
assert math.isclose(float(kw12[1]), 1.0, abs_tol=1e-4)
print("pass winding_factor_spectrum")

print()
print("All winding factor tests passed")

# ── Parameterised spectrum checks ─────────────────────────────────────────────
_spectrum_combos = [
    # (Q,  P, layers, nu_working, kw_expected, tol, label)
    (12, 10, 2, 5, 0.9330, 1e-3, '12s/10p FSCW'),
    (12, 14, 2, 7, 0.9330, 1e-3, '12s/14p FSCW'),
    (12,  8, 2, 4, 0.8660, 1e-3, '12s/8p  FSCW'),
    (24,  4, 2, 2, 0.9659, 1e-3, '24s/4p  ISW'),
    (36,  8, 2, 4, 0.9452, 1e-3, '36s/8p  ISW'),
]

for Q, P, layers, nu_w, kw_exp, tol, label in _spectrum_combos:
    nu_arr, kw_arr = winding_factor_spectrum(Q, P, layers=layers, nu_max=nu_w*2)  # best span
    kw_val = float(kw_arr[nu_w - 1])
    assert math.isclose(kw_val, kw_exp, abs_tol=tol),         f"{label}: kw(nu={nu_w})={kw_val:.4f}, expected {kw_exp}"
    assert np.all(kw_arr >= -1e-10) and np.all(kw_arr <= 1.0 + 1e-10),         f"{label}: kw out of [0,1]"
    print(f"pass  {label}: kw(nu={nu_w})={kw_val:.4f}")
